# Scenario 1: Customer Support Resolution Agent

**Domains tested:** Agentic Architecture (D1 -- 27%), Tool Design & MCP (D2 -- 18%), Context Management (D5 -- 15%)

---

## Scenario Context

You are the lead engineer at **ShipFast**, an e-commerce logistics company. Your team is building a customer support agent using the Anthropic API with tool use. The agent must handle order inquiries, process refunds, and escalate complex cases to human agents.

**Production requirements:**
- 80%+ first-contact resolution rate
- Refunds can only be processed after verifying order exists AND is within the 30-day return window
- The agent must escalate to a human when: (a) the customer is angry after 2 failed resolution attempts, (b) the issue involves a safety concern, or (c) the refund exceeds $500
- All escalations must include a structured handoff summary so the human agent has full context
- Multi-issue conversations must preserve context across topic switches

**MCP tools available:**
- `get_customer(customer_id)` -- retrieves customer profile and history
- `lookup_order(order_id)` -- retrieves order details including status, date, items, total
- `process_refund(order_id, reason, amount)` -- issues refund to original payment method
- `escalate_to_human(summary, priority, category)` -- transfers to human agent queue

---

In [ ]:
import anthropic
import json
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

---

## Part 1: Exam-Format Questions

Answer each question, then reveal the explanation. These questions span multiple domains, just like the real exam.

---

### Question 1 (D1 + D2)

The customer support agent has tools `lookup_order` and `process_refund`. A customer requests a refund for order #12345. The agent calls `process_refund` immediately without first calling `lookup_order`.

In production, the refund fails because the order is outside the 30-day return window. What is the best architectural fix?

- **A)** Implement programmatic prerequisites in the tool orchestration layer so `process_refund` cannot execute unless `lookup_order` has been called first and the return window is validated
- **B)** Add a system prompt instruction: "Always look up orders before processing refunds"
- **C)** Add retry logic so the agent tries the refund again after looking up the order
- **D)** Use a higher temperature to encourage the model to explore more careful workflows

In [ ]:
# Reveal answer
q1 = {
    "correct": "A",
    "explanation": (
        "Programmatic prerequisites enforce the constraint at the code level, not the prompt level. "
        "Option B relies on the model following instructions 100% of the time -- which is not guaranteed, "
        "especially under adversarial or unusual inputs. The exam tests whether you know that critical "
        "business rules (like 'verify before refund') must be enforced programmatically, not via prompt. "
        "This is a core Domain 1 (Agentic Architecture) concept: never trust the model to enforce "
        "business-critical constraints. Option C is reactive, not preventive. Option D is nonsensical."
    )
}
print(f"Correct: {q1['correct']}")
print(f"\nExplanation: {q1['explanation']}")

### Question 2 (D2 + D5)

A customer contacts support about two issues in the same conversation: (1) a missing item from order #A, and (2) a billing question about order #B. After resolving the first issue, the agent starts discussing order #B but references details from order #A.

What is the root cause?

- **A)** The model's context window is too small to hold both orders
- **B)** The model is hallucinating order details because it forgot the second lookup
- **C)** The conversation should be split into two separate sessions, one per issue
- **D)** The tool results from `lookup_order(#A)` are still in the conversation context, and the agent lacks explicit instructions to scope tool results to the current issue

In [ ]:
q2 = {
    "correct": "D",
    "explanation": (
        "When tool results accumulate in the conversation, the model may reference stale data from "
        "earlier tool calls. The fix is explicit context scoping: either (1) instruct the agent to "
        "always call lookup_order for the current issue before answering, or (2) structure the "
        "conversation to clearly delineate issue boundaries. Option A is unlikely -- two order lookups "
        "easily fit in context. Option B is possible but the root cause is architectural, not model "
        "capability. Option C breaks the customer experience. This tests Domain 5 (Context Management) "
        "-- specifically, how tool results in conversation history create context pollution."
    )
}
print(f"Correct: {q2['correct']}")
print(f"\nExplanation: {q2['explanation']}")

### Question 3 (D1 + D2)

Your escalation tool has this signature: `escalate_to_human(summary: str, priority: str, category: str)`. In production, human agents report that escalation summaries are inconsistent -- sometimes they get a one-sentence summary, sometimes a page of raw conversation. What is the best fix?

- **A)** Add "write a detailed summary" to the system prompt
- **B)** Replace the `summary` string parameter with a structured schema requiring `customer_id`, `issue_description`, `steps_attempted`, `resolution_status`, and `customer_sentiment`
- **C)** Use few-shot examples in the system prompt showing ideal summaries
- **D)** Post-process the summary with a second Claude call to standardize format

In [ ]:
q3 = {
    "correct": "B",
    "explanation": (
        "Structured tool schemas force the model to produce specific fields, eliminating inconsistency "
        "at the structural level. Option C (few-shot examples) can help but is less reliable than "
        "schema enforcement -- the model can still deviate from examples. Option A is vague. Option D "
        "adds latency and cost without addressing the root cause. This tests Domain 2 (Tool Design): "
        "structured output schemas are more reliable than prompt-based formatting instructions. "
        "The exam frequently tests whether you choose schema enforcement over prompt instructions "
        "for output consistency."
    )
}
print(f"Correct: {q3['correct']}")
print(f"\nExplanation: {q3['explanation']}")

### Question 4 (D1 + D5)

Your customer support agent handles an average of 8 tool calls per conversation. After 6 months in production, you notice the agent's accuracy degrades significantly in conversations with more than 12 tool calls. The most common failure: the agent re-asks questions the customer already answered.

What is the most effective mitigation?

- **A)** Summarize resolved issues periodically and replace earlier conversation turns with the summary to keep the working context focused
- **B)** Increase the max_tokens parameter to give the model more room to process
- **C)** Switch to a model with a larger context window
- **D)** Limit conversations to 10 tool calls and force escalation after that

In [ ]:
q4 = {
    "correct": "A",
    "explanation": (
        "Context window degradation is a key Domain 5 concept. Even when conversation fits in the "
        "context window, long conversations degrade attention to earlier details. Periodic summarization "
        "compresses resolved topics and keeps the working context focused on the current issue. "
        "Option B confuses output tokens with input context. Option C may help but doesn't address "
        "the attention degradation problem -- a larger window with the same noise still loses details. "
        "Option D artificially limits the agent's capability. The exam tests whether you understand "
        "that context window size and context quality are different problems."
    )
}
print(f"Correct: {q4['correct']}")
print(f"\nExplanation: {q4['explanation']}")

### Question 5 (D2)

Your `process_refund` tool returns `{"status": "error", "code": "PAYMENT_PROVIDER_TIMEOUT"}`. The agent tells the customer: "Your refund has been denied." What architectural change prevents this misinterpretation?

- **A)** Add error handling instructions to the system prompt: "If you see an error, tell the customer to try again later"
- **B)** Catch the error in application code and never pass it to the agent
- **C)** Return structured error responses with explicit `user_message` and `is_retryable` fields so the agent does not need to interpret raw error codes
- **D)** Train the model to understand payment provider error codes

In [ ]:
q5 = {
    "correct": "C",
    "explanation": (
        "Structured error responses with human-readable messages and retryability flags remove the "
        "need for the model to interpret technical error codes. The agent can relay the user_message "
        "directly and decide whether to retry based on is_retryable. Option A helps but is fragile -- "
        "the model may not map all error codes correctly. Option B prevents the agent from informing "
        "the customer at all. Option D is not feasible. This tests Domain 2 (Tool Design): tool "
        "responses should be designed for the consumer (the model), not just the system."
    )
}
print(f"Correct: {q5['correct']}")
print(f"\nExplanation: {q5['explanation']}")

### Question 6 (D1 + D2 + D5)

A customer asks for a refund on a $750 order. Your agent has the following escalation rules in its system prompt:

> "Escalate to a human agent when the refund amount exceeds $500."

However, in production, the agent sometimes processes the $750 refund directly without escalating. After investigating, you find this happens when the conversation is long and the system prompt instruction gets "diluted" by many tool results.

What is the most reliable fix?

- **A)** Repeat the escalation rule at the end of every assistant message
- **B)** Use prompt caching to ensure the system prompt stays in the model's attention
- **C)** Move the $500 threshold check from the system prompt to the application code -- check the refund amount programmatically before executing `process_refund`
- **D)** Shorten the system prompt so the escalation rule is more prominent

In [ ]:
q6 = {
    "correct": "C",
    "explanation": (
        "This is the same principle as Question 1: critical business rules must be enforced "
        "programmatically. When the refund amount exceeds $500, your application code should intercept "
        "the process_refund call and route to escalation instead. The model should not be the sole "
        "gatekeeper for financial thresholds. Option A is noisy and still relies on the model. "
        "Option B (prompt caching) helps with cost, not with attention degradation in long "
        "conversations. Option D may help marginally but does not solve the fundamental issue. "
        "This combines Domain 1 (never trust model for business rules), Domain 2 (tool-level "
        "enforcement), and Domain 5 (context dilution in long conversations)."
    )
}
print(f"Correct: {q6['correct']}")
print(f"\nExplanation: {q6['explanation']}")

### Question 7 (D1 + D2)

You want the agent to use few-shot examples for escalation decisions. Which placement produces the most consistent escalation behavior?

- **A)** In the user message, before the customer's actual request
- **B)** In the system prompt, as part of the escalation tool's description
- **C)** In a separate tool call that the agent retrieves on demand
- **D)** As synthetic conversation turns (user/assistant pairs) at the beginning of the messages array, before the real conversation

In [ ]:
q7 = {
    "correct": "D",
    "explanation": (
        "Synthetic conversation turns (few-shot examples as user/assistant message pairs) are the "
        "most effective placement for behavioral examples. They show the model exactly how to behave "
        "in context, as if it had already handled similar cases. Option B (tool description) works "
        "for output format but is limited in space and less effective for complex decision logic. "
        "Option A mixes examples with the actual request, which can confuse the model. Option C "
        "adds latency and may not be retrieved when needed. This tests Domain 1 (agentic patterns) "
        "and Domain 2 (tool-adjacent prompting techniques)."
    )
}
print(f"Correct: {q7['correct']}")
print(f"\nExplanation: {q7['explanation']}")

---

## Part 2: Hands-On Build

Now implement the customer support agent described in the scenario. We will simulate the MCP tools with local functions and wire up the agentic loop.

---

### Step 1: Define the simulated tools and data

In [ ]:
# Simulated database
CUSTOMERS = {
    "CUST-001": {
        "name": "Alice Johnson",
        "email": "alice@example.com",
        "tier": "gold",
        "history": ["ORD-100", "ORD-101", "ORD-102"]
    },
    "CUST-002": {
        "name": "Bob Smith",
        "email": "bob@example.com",
        "tier": "standard",
        "history": ["ORD-200"]
    }
}

ORDERS = {
    "ORD-100": {
        "customer_id": "CUST-001",
        "status": "delivered",
        "date": (datetime.now() - timedelta(days=10)).isoformat(),
        "items": [{"name": "Wireless Headphones", "price": 89.99, "qty": 1}],
        "total": 89.99,
        "payment_method": "credit_card"
    },
    "ORD-101": {
        "customer_id": "CUST-001",
        "status": "delivered",
        "date": (datetime.now() - timedelta(days=45)).isoformat(),
        "items": [{"name": "Laptop Stand", "price": 149.99, "qty": 1}],
        "total": 149.99,
        "payment_method": "credit_card"
    },
    "ORD-102": {
        "customer_id": "CUST-001",
        "status": "shipped",
        "date": (datetime.now() - timedelta(days=2)).isoformat(),
        "items": [{"name": "USB-C Hub", "price": 59.99, "qty": 2}],
        "total": 119.98,
        "payment_method": "debit_card"
    },
    "ORD-200": {
        "customer_id": "CUST-002",
        "status": "delivered",
        "date": (datetime.now() - timedelta(days=5)).isoformat(),
        "items": [{"name": "Standing Desk", "price": 749.99, "qty": 1}],
        "total": 749.99,
        "payment_method": "credit_card"
    }
}

# Track state for programmatic prerequisites
refund_prerequisites = {}  # order_id -> {"looked_up": bool, "within_window": bool}
escalation_log = []  # list of escalation summaries


def get_customer(customer_id: str) -> dict:
    """Retrieve customer profile."""
    if customer_id in CUSTOMERS:
        return {"status": "success", "data": CUSTOMERS[customer_id]}
    return {"status": "error", "code": "CUSTOMER_NOT_FOUND",
            "user_message": f"No customer found with ID {customer_id}"}


def lookup_order(order_id: str) -> dict:
    """Retrieve order details and check refund eligibility."""
    if order_id not in ORDERS:
        return {"status": "error", "code": "ORDER_NOT_FOUND",
                "user_message": f"No order found with ID {order_id}"}
    order = ORDERS[order_id]
    order_date = datetime.fromisoformat(order["date"])
    days_since = (datetime.now() - order_date).days
    within_window = days_since <= 30

    # Record prerequisite check
    refund_prerequisites[order_id] = {
        "looked_up": True,
        "within_window": within_window
    }

    return {
        "status": "success",
        "data": {**order, "order_id": order_id,
                 "days_since_order": days_since,
                 "refund_eligible": within_window}
    }


def process_refund(order_id: str, reason: str, amount: float) -> dict:
    """Process refund with programmatic prerequisite checks."""
    # PROGRAMMATIC PREREQUISITE: must have looked up the order first
    prereqs = refund_prerequisites.get(order_id)
    if not prereqs or not prereqs.get("looked_up"):
        return {
            "status": "error",
            "code": "PREREQUISITE_NOT_MET",
            "user_message": "Order must be looked up before processing a refund.",
            "is_retryable": True
        }

    # PROGRAMMATIC CHECK: return window
    if not prereqs.get("within_window"):
        return {
            "status": "error",
            "code": "OUTSIDE_RETURN_WINDOW",
            "user_message": "This order is outside the 30-day return window and is not eligible for a refund.",
            "is_retryable": False
        }

    # PROGRAMMATIC CHECK: amount threshold for escalation
    if amount > 500:
        return {
            "status": "error",
            "code": "AMOUNT_REQUIRES_ESCALATION",
            "user_message": f"Refunds over $500 require human agent approval. Please escalate this case.",
            "is_retryable": False
        }

    return {
        "status": "success",
        "data": {"refund_id": f"REF-{order_id}", "amount": amount,
                 "method": ORDERS[order_id]["payment_method"],
                 "estimated_days": 5}
    }


def escalate_to_human(summary: dict) -> dict:
    """Escalate to human agent with structured summary."""
    required_fields = ["customer_id", "issue_description", "steps_attempted",
                       "resolution_status", "customer_sentiment"]
    missing = [f for f in required_fields if f not in summary]
    if missing:
        return {
            "status": "error",
            "code": "INCOMPLETE_SUMMARY",
            "user_message": f"Escalation summary missing required fields: {missing}",
            "is_retryable": True
        }
    escalation_log.append(summary)
    return {
        "status": "success",
        "data": {"ticket_id": f"ESC-{len(escalation_log):03d}",
                 "queue_position": 3,
                 "estimated_wait": "5 minutes"}
    }


# Dispatch table
TOOL_DISPATCH = {
    "get_customer": lambda args: get_customer(args["customer_id"]),
    "lookup_order": lambda args: lookup_order(args["order_id"]),
    "process_refund": lambda args: process_refund(
        args["order_id"], args["reason"], args["amount"]
    ),
    "escalate_to_human": lambda args: escalate_to_human(args["summary"]),
}

print("Simulated tools loaded.")
print(f"Customers: {list(CUSTOMERS.keys())}")
print(f"Orders: {list(ORDERS.keys())}")

### Step 2: Define the tool schemas for Claude

In [ ]:
TOOLS = [
    {
        "name": "get_customer",
        "description": "Retrieve a customer's profile including name, email, tier, and order history. Use this to identify the customer and understand their account status.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "The customer ID (e.g., CUST-001)"
                }
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "lookup_order",
        "description": "Retrieve order details including status, items, total, and refund eligibility. MUST be called before process_refund to verify the order exists and is within the return window.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID (e.g., ORD-100)"
                }
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "process_refund",
        "description": "Issue a refund for an order. Prerequisites: lookup_order must have been called first and the order must be within the 30-day return window. Refunds over $500 require escalation to a human agent.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID to refund"
                },
                "reason": {
                    "type": "string",
                    "description": "The reason for the refund"
                },
                "amount": {
                    "type": "number",
                    "description": "The refund amount in dollars"
                }
            },
            "required": ["order_id", "reason", "amount"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Transfer the conversation to a human agent. Requires a structured summary with all required fields.",
        "input_schema": {
            "type": "object",
            "properties": {
                "summary": {
                    "type": "object",
                    "description": "Structured handoff summary",
                    "properties": {
                        "customer_id": {"type": "string"},
                        "issue_description": {"type": "string"},
                        "steps_attempted": {"type": "string"},
                        "resolution_status": {"type": "string"},
                        "customer_sentiment": {"type": "string"}
                    },
                    "required": ["customer_id", "issue_description",
                                 "steps_attempted", "resolution_status",
                                 "customer_sentiment"]
                }
            },
            "required": ["summary"]
        }
    }
]

print(f"Defined {len(TOOLS)} tool schemas.")

### Step 3: Build the agentic loop

In [ ]:
SYSTEM_PROMPT = """You are a customer support agent for ShipFast, an e-commerce logistics company.

RULES:
1. Always greet the customer by name after looking up their profile.
2. Before processing any refund, you MUST call lookup_order first to verify eligibility.
3. Escalate to a human agent when:
   - The customer expresses strong frustration after two failed resolution attempts
   - The issue involves a safety concern
   - The refund amount exceeds $500
4. When escalating, provide a complete structured summary.
5. For multi-issue conversations, address each issue separately and confirm resolution before moving on.

TONE: Professional, empathetic, solution-oriented. Acknowledge frustration before offering solutions."""


def run_agent(user_message: str, conversation: list = None, max_turns: int = 10):
    """Run the agentic loop with tool use."""
    if conversation is None:
        conversation = []

    conversation.append({"role": "user", "content": user_message})

    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=conversation
        )

        # Append assistant response
        conversation.append({"role": "assistant", "content": response.content})

        # If the model wants to use tools, execute them
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_name = block.name
                    tool_input = block.input
                    print(f"  [TOOL] {tool_name}({json.dumps(tool_input, indent=2)[:200]})")

                    # Execute the tool
                    if tool_name in TOOL_DISPATCH:
                        result = TOOL_DISPATCH[tool_name](tool_input)
                    else:
                        result = {"status": "error", "code": "UNKNOWN_TOOL",
                                  "user_message": f"Unknown tool: {tool_name}"}

                    print(f"  [RESULT] {json.dumps(result)[:200]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })

            conversation.append({"role": "user", "content": tool_results})
        else:
            # Model finished -- extract text response
            text_response = ""
            for block in response.content:
                if hasattr(block, "text"):
                    text_response += block.text
            print(f"\n  [AGENT]: {text_response}")
            return text_response, conversation

    return "Max turns reached.", conversation


print("Agent loop ready.")

### Step 4: Test -- Happy path (eligible refund)

In [ ]:
print("=== Test 1: Happy path refund ===")
print("Customer CUST-001 requests refund for ORD-100 (delivered 10 days ago, $89.99)\n")

refund_prerequisites.clear()  # reset state

response, conv = run_agent(
    "Hi, I'm customer CUST-001. I'd like a refund for order ORD-100. "
    "The headphones arrived but they don't fit well."
)

print("\n--- Prerequisite check ---")
print(f"Was lookup_order called first? {'ORD-100' in refund_prerequisites}")
print(f"Refund prerequisites state: {refund_prerequisites}")

### Step 5: Test -- Programmatic prerequisite blocks expired refund

In [ ]:
print("=== Test 2: Refund outside return window ===")
print("Customer CUST-001 requests refund for ORD-101 (delivered 45 days ago)\n")

refund_prerequisites.clear()

response, conv2 = run_agent(
    "Hi, I'm customer CUST-001. I need a refund for order ORD-101. "
    "The laptop stand broke after a month of use."
)

print("\n--- Prerequisite check ---")
print(f"Refund prerequisites state: {refund_prerequisites}")
print(f"Within window: {refund_prerequisites.get('ORD-101', {}).get('within_window', 'N/A')}")

### Step 6: Test -- Escalation for high-value refund

In [ ]:
print("=== Test 3: Escalation for $749.99 refund ===")
print("Customer CUST-002 requests refund for ORD-200 ($749.99 standing desk)\n")

refund_prerequisites.clear()
escalation_log.clear()

response, conv3 = run_agent(
    "I'm customer CUST-002. I want a full refund for order ORD-200. "
    "The standing desk wobbles dangerously and I'm worried it might collapse."
)

print("\n--- Escalation check ---")
print(f"Escalations logged: {len(escalation_log)}")
if escalation_log:
    print(f"Escalation summary: {json.dumps(escalation_log[-1], indent=2)}")

---

## Part 3: Failure Injections

Deliberately introduce problems and diagnose them. This mirrors the exam's "diagnose the production problem" pattern.

---

### Failure 1: Remove programmatic prerequisites

In [ ]:
# What happens if we remove the prerequisite check from process_refund?

def process_refund_NO_PREREQ(order_id: str, reason: str, amount: float) -> dict:
    """Refund WITHOUT prerequisite checks -- the wrong way."""
    # No check for lookup_order, no window check, no amount threshold
    if order_id not in ORDERS:
        return {"status": "error", "code": "ORDER_NOT_FOUND",
                "user_message": "Order not found."}
    return {
        "status": "success",
        "data": {"refund_id": f"REF-{order_id}", "amount": amount,
                 "method": ORDERS[order_id]["payment_method"]}
    }


# Temporarily replace the dispatch
original_dispatch = TOOL_DISPATCH["process_refund"]
TOOL_DISPATCH["process_refund"] = lambda args: process_refund_NO_PREREQ(
    args["order_id"], args["reason"], args["amount"]
)

print("=== FAILURE INJECTION: No prerequisites ===")
print("Watch: the agent may process a refund for an expired order or a $750 order\n")

refund_prerequisites.clear()
response, conv_fail = run_agent(
    "I'm CUST-002. Refund ORD-200 immediately. The desk is dangerous."
)

print("\n--- DIAGNOSIS ---")
print("Without programmatic prerequisites, the $749.99 refund may be processed")
print("directly -- bypassing the $500 escalation threshold.")
print("The system prompt says to escalate, but the model is not guaranteed to follow it.")
print("This is why critical business rules MUST be enforced in code, not prompts.")

# Restore
TOOL_DISPATCH["process_refund"] = original_dispatch

### Failure 2: Unstructured escalation summaries

In [ ]:
# What happens if the escalation tool accepts a plain string instead of a structured object?

TOOLS_UNSTRUCTURED = TOOLS.copy()
TOOLS_UNSTRUCTURED[3] = {
    "name": "escalate_to_human",
    "description": "Transfer to human agent. Include a summary of the issue.",
    "input_schema": {
        "type": "object",
        "properties": {
            "summary": {
                "type": "string",
                "description": "Summary of the issue"
            },
            "priority": {
                "type": "string",
                "description": "Priority level"
            }
        },
        "required": ["summary"]
    }
}

print("=== FAILURE INJECTION: Unstructured escalation ===")
print("Using a plain string 'summary' instead of a structured object.")
print("")
print("With structured schema, the model MUST provide:")
print("  - customer_id, issue_description, steps_attempted,")
print("    resolution_status, customer_sentiment")
print("")
print("With unstructured string, the model might produce:")
print('  - "Customer wants a refund for a desk"')
print('  - "CUST-002 ordered a standing desk that wobbles. They want a refund...(3 paragraphs)"')
print("")
print("Inconsistent summaries slow down human agents and reduce resolution speed.")
print("LESSON: Use structured schemas to enforce output consistency (Domain 2).")

### Failure 3: Context pollution in multi-issue conversations

In [ ]:
print("=== FAILURE INJECTION: Context pollution ===")
print("Simulating a multi-issue conversation where order data accumulates.\n")

refund_prerequisites.clear()

# First issue
response1, multi_conv = run_agent(
    "Hi, I'm CUST-001. I have two issues. First, can you check on order ORD-100?"
)

print("\n--- Now switching to second issue ---\n")

# Second issue -- the agent now has ORD-100 data in context
response2, multi_conv = run_agent(
    "OK thanks. Now my second issue: what's the status of order ORD-102?",
    conversation=multi_conv
)

print("\n--- DIAGNOSIS ---")
print("Both ORD-100 and ORD-102 tool results are now in the conversation.")
print("If the agent later references details, it might mix up data between orders.")
print("Mitigation: scope tool results to the current issue, or periodically")
print("summarize resolved issues and trim earlier context (Domain 5).")

---

## Key Takeaways

| Concept | Domain | Lesson |
|---------|--------|--------|
| Programmatic prerequisites | D1, D2 | Critical business rules must be enforced in code, never solely via prompts |
| Structured tool schemas | D2 | Use schemas (not free text) to enforce consistent tool inputs and outputs |
| Structured error responses | D2 | Include `user_message` and `is_retryable` so the agent interprets errors correctly |
| Few-shot escalation examples | D1 | Synthetic conversation turns are the most effective placement for behavioral examples |
| Context scoping | D5 | In multi-issue conversations, tool results accumulate and can cause cross-contamination |
| Context summarization | D5 | Periodically compress resolved issues to keep the working context focused |
| Escalation thresholds in code | D1 | Financial and safety thresholds must be checked programmatically, not by the model |